In [1]:
import sys
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path
import kagglehub

c:\Users\BIT\Desktop\Deep_Generative_Modelling\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
current_dir = Path().cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("done")

done


In [4]:
data_path = project_root / "data"

In [ ]:
# cache_path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")
# print("Path to dataset files:", cache_path)

# for item in Path(cache_path).iterdir():
#     shutil.copy2(item, data_path/item.name)
    
# print("Data Saved")

100%|██████████| 195M/195M [00:41<00:00, 4.98MB/s] 

Extracting files...


Path to dataset files: C:\Users\BIT\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1
Data Saved


In [ ]:
# for file in data_path.iterdir():
#     print(str(file))

c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\Exam_Score_Prediction.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_scores.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_tags.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\link.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\movie.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\rating.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\tag.csv


In [5]:
movie_df = pd.read_csv(str(data_path / "movie.csv"))
rating_df = pd.read_csv(str(data_path / "rating.csv"))

In [6]:
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [8]:
movie_df.shape, rating_df.shape

((27278, 3), (20000263, 4))

In [9]:
rating_df = rating_df.iloc[0:1000000]

In [10]:
len(rating_df["userId"].unique())

6743

In [11]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [12]:
rating_df.isna().all()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [13]:
movie_df.isna().all()

movieId    False
title      False
genres     False
dtype: bool

In [14]:
# Let's create a pivot table to better understand the data
pivot_ratings_df = rating_df.pivot(index="userId", columns="movieId", values="rating")

In [15]:
pivot_ratings_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
pivot_ratings_df.shape

(6743, 13950)

In [37]:
pivot_ratings_df_cy = pivot_ratings_df.copy()

In [38]:
# def binaryconvertor(rating_df):
#     if rating_df <= 2:
#         return 0
#     else:
#         return 1

In [39]:
# # pivot_ratings_df.applymap(lambda x: binaryconvertor(x) if pd.notna(x) else x)
# # let's use numpy
# mask = pivot_ratings_df.notna()
# pivot_ratings_df[mask] = pivot_ratings_df[mask].apply(binaryconvertor)

In [40]:
pivot_ratings_df_cy = pd.DataFrame(
    np.where(pivot_ratings_df_cy.isna(), -1, (pivot_ratings_df_cy > 2).astype(int)),
    index=pivot_ratings_df_cy.index,
    columns=pivot_ratings_df_cy.columns
)

In [41]:
pivot_ratings_df_cy.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
2,-1,-1,1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
3,1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
4,-1,-1,-1,-1,-1,1,-1,-1,-1,1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
5,-1,1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
6,1,-1,1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
7,-1,-1,1,-1,-1,-1,1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
8,1,-1,1,-1,-1,1,-1,-1,-1,1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
9,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,...,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [42]:
pivot_ratings_df_cy.shape

(6743, 13950)

In [43]:
pivot_ratings_df_cy.index

Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
       6734, 6735, 6736, 6737, 6738, 6739, 6740, 6741, 6742, 6743],
      dtype='int64', name='userId', length=6743)

In [44]:
pivot_ratings_df.isna().all()

movieId
1         False
2         False
3         False
4         False
5         False
          ...  
130073    False
130219    False
130462    False
130490    False
130642    False
Length: 13950, dtype: bool

Starting to build the RBM

In [45]:
NV = pivot_ratings_df_cy.shape[1]
NH = 100
BATCH_SIZE = 100

In [46]:
weigths = np.random.standard_normal(size=(NV, NH))
a = np.zeros((1, NH))
b = np.zeros((1, NV))

In [47]:
def sigmoid(mat):
    return 1/(1 + (np.exp(-mat)))

In [48]:
# testing it
sample = np.ones((3,4))
sigmoid(sample)

array([[0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858]])

In [49]:
def sample_h(sample, weigths, biases):
    activation = sample @ weigths + biases
    prob_h = sigmoid(activation)
    binary = np.random.binomial(n=1, p=prob_h)
    return prob_h, binary 

In [50]:
def sample_v(input, weigths, baises):
    activation = input @ weigths.T + baises
    prob_v = sigmoid(activation)
    binary = np.random.binomial(n=1, p=prob_v)
    return prob_v, binary

In [51]:
# We will start training
nb_epoch = 10
learning_rate = 0.01

In [73]:
def contrastive_divergence(v0: np.array, weigths: np.array, a: np.array, b: np.array, mask: np.array, learning_rate: float):
    """Performs onr step of CD

    Args:
        v0 (np.array): batch of real data
        weigths (np.array): weigths of the RBM
        a (np.array): biases for hidden layer
        b (np.array): biases for visual layer
        mask (np.array): mask array for missing values
        learning_rate (float): learning_rate
        
    Returns:
        updated_weigths, updated_a, updated_b, reconstruction_error
    """
    
    # positive phase
    prob_h0, h0_binary = sample_h(v0, weigths, a)
    
    # negative phase
    prob_v1, v1_sampled = sample_v(h0_binary, weigths, b)
    v1_final = np.where(mask, v1_sampled, v0)
    
    # Sample hidden from reconstructed visible
    prob_h1, h1_binary = sample_h(v1_final, weigths, a)
    
    # Computing gradients
    positive_grad = (v0.T @ prob_h0) / v0.shape[0]
    negative_grad = (v1_final.T @ prob_h1) / v0.shape[0]
    
    # Updating the weigths and biases
    weigths_update = learning_rate * (positive_grad - negative_grad)
    new_weigths = weigths + weigths_update
    
    a_update = learning_rate * np.mean(prob_h0 - prob_h1, axis=0)
    new_a = a + a_update
    
    difference = (v0 - v1_final) * mask
    b_update = learning_rate * np.sum(difference, axis=0) / np.sum(mask, axis=0, keepdims=True)
    new_b = b + b_update
    
    # Calculate reconstruction error
    error = np.sum((v0 - v1_final) ** 2 * mask) / np.sum(mask)
    
    return new_weigths, new_a, new_b, error

In [74]:
indexes = np.array(pivot_ratings_df_cy.index)
np.random.shuffle(indexes)

In [75]:
indexes

array([5557, 2130, 1028, ..., 4686, 2693, 4643], shape=(6743,))

In [81]:
def train_rbm(data: np.array, mask: np.array, weigths: np.array, a: np.array, b: np.array, epochs:int, batch_size: int, learning_rate: float):
    """Function to train the RBM.

    Args:
        data (np.array): Training Data
        mask (np.array): mask for missing values
        weigths (np.array): weighths of the RBM
        a (np.array): biases for hidden layer
        b (np.array): biases for visual layer
        epochs (int): No of epochs to train
        batch_size (int): batch size
        learning_rate (float): learning rate
    """
    
    num_samples = data.shape[0]
    num_batches = num_samples // batch_size
    error_history = []
    
    for epoch in tqdm(range(epochs)):
        epoch_error = 0
        indexes = np.arange(num_samples)
        np.random.shuffle(indexes)
        
        shuffled_data = data[indexes]
        shuffled_mask = mask[indexes]
        
        for batch_idx in range(num_batches):
            start_idx = batch_idx * batch_size
            end_idx = start_idx + batch_size
            
            batch_data = shuffled_data[start_idx : end_idx]
            batch_mask = shuffled_mask[start_idx : end_idx]
            
            # perform CD
            weigths, a, b, batch_error = contrastive_divergence(
                batch_data, weigths, a, b, batch_mask, learning_rate
            )
            
            epoch_error += batch_error
            
        avg_error = epoch_error / num_batches
        error_history.append(avg_error)
        
        # Print progress
        if epoch % 1 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Error: {avg_error:.4f}")
        
    return weigths, a, b, error_history

In [83]:
mask = pivot_ratings_df_cy < 0
# mask.loc[[1,2,3]]

In [84]:
train_rbm(
    data=pivot_ratings_df_cy.values,
    mask=mask.values,
    weigths=weigths,
    a=a,
    b=b,
    epochs=10,
    batch_size=BATCH_SIZE,
    learning_rate=0.01
)

  0%|          | 0/10 [00:00<?, ?it/s]C:\Users\BIT\AppData\Local\Temp\ipykernel_1108\3716844040.py:2: RuntimeWarning: overflow encountered in exp
  return 1/(1 + (np.exp(-mat)))
 10%|█         | 1/10 [00:20<03:03, 20.40s/it]

Epoch 1/10, Error: 1.1368


 20%|██        | 2/10 [00:39<02:37, 19.72s/it]

Epoch 2/10, Error: 1.0000


 30%|███       | 3/10 [00:58<02:15, 19.31s/it]

...

 90%|█████████ | 9/10 [02:50<00:18, 18.81s/it]

Epoch 9/10, Error: 1.0000


100%|██████████| 10/10 [03:09<00:00, 18.90s/it]

Epoch 10/10, Error: 1.0000


(array([[-1.62461657, -0.31823278, -2.53309569, ..., -1.71882829,
         -2.39641099, -2.43725179],
        [-4.40689209, -5.38175939, -4.6496745 , ..., -4.54775922,
         -4.34005355, -4.26900197],
        [-4.62139   , -6.13036426, -7.3387726 , ..., -5.45182718,
         -5.0545997 , -4.79113695],
        ...,
        [-8.99029517, -5.88062482, -4.89016903, ..., -5.90528974,
         -7.43905614, -7.4661701 ],
        [-4.96707716, -6.35590222, -6.5230384 , ..., -7.95187878,
         -4.95451669, -7.23110186],
        [-3.71776335, -6.9023426 , -7.20812358, ..., -7.84756036,
         -6.89653497, -8.53565305]], shape=(13950, 100)),
 array([[6.49733823, 6.5735092 , 6.57149095, 6.57517831, 6.53054295,
         6.55449803, 6.32871691, 6.59909258, 6.56675309, 6.54514327,
         5.88505268, 6.53556541, 6.55341639, 6.5650854 , 6.49841221,
         6.57141193, 6.54945994, 6.4200605 , 6.57690267, 6.59220338,
         6.55137324, 6.59512277, 6.53442332, 6.51425266, 6.52418115,
        